In [1]:
# [Markdown]
# # 🧠 Lab 02: Multi-Task Alpha Engine & Wavelet Denoising
# **Upgrades Applied:**
# 1. Wavelet Denoising (Zero-lag noise filtering)
# 2. Multi-Task Quantformer (Predicts Direction, Magnitude, and Volatility)
# 3. Combined Financial Loss (BCE + MSE + Sharpe Proxy)

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib
import pywt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from IPython.display import display

# --- 1. WAVELET DENOISING ALGORITHM ---
def denoise_series(data, wavelet='db4', level=2):
    """Removes high-frequency micro-chop from price data without adding lag."""
    coeffs = pywt.wavedec(data, wavelet, mode='smooth', level=level)
    # Zero out the highest frequency details (noise)
    coeffs[-1] = np.zeros_like(coeffs[-1])
    coeffs[-2] = np.zeros_like(coeffs[-2])
    return pywt.waverec(coeffs, wavelet)[:len(data)]

# --- 2. MULTI-TASK DATASET ---
class MultiTaskDataset(Dataset):
    def __init__(self, X, y_dir, y_mag, y_vol, seq_len):
        self.X, self.y_dir, self.y_mag, self.y_vol, self.seq_len = X, y_dir, y_mag, y_vol, seq_len
        
    def __len__(self): return len(self.X) - self.seq_len
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx : idx + self.seq_len], dtype=torch.float32), 
                torch.tensor(self.y_dir[idx + self.seq_len - 1], dtype=torch.float32),
                torch.tensor(self.y_mag[idx + self.seq_len - 1], dtype=torch.float32),
                torch.tensor(self.y_vol[idx + self.seq_len - 1], dtype=torch.float32))

# --- 3. MULTI-TASK QUANTFORMER ---
class MultiTaskQuantformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(num_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, batch_first=True, dropout=0.2
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        
        # Shared representations
        self.fc_shared = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.2))
        
        # Three distinct "Heads"
        self.head_dir = nn.Linear(32, 1) # Probability of UP
        self.head_mag = nn.Linear(32, 1) # Expected Return
        self.head_vol = nn.Sequential(nn.Linear(32, 1), nn.Softplus()) # Expected Volatility (must be > 0)
        
    def forward(self, x):
        shared_features = self.fc_shared(self.transformer(self.proj(x))[:, -1, :])
        return self.head_dir(shared_features), self.head_mag(shared_features), self.head_vol(shared_features)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"⚡ Compute Device: {device}")

⚡ Compute Device: mps


In [3]:
# [Markdown]
# ### 1. Data Ingestion & Target Generation
# We apply the DWT (Discrete Wavelet Transform) to the close price, then build the 3 targets.

df = pd.read_parquet('../data/processed/clean_tensor.parquet')

print("🌊 Applying Wavelet Denoising to Price Action...")
# .to_numpy(copy=True) forces a writeable array in memory
df['clean_close'] = denoise_series(df['close'].to_numpy(copy=True))

# Best Grid Search Params
H = 4   # 1-Hour Target Horizon
S = 32  # 8-Hour Context Memory (Reverted to S=32 to avoid the 'Scalper Trap')

# --- GENERATE MULTI-TASK TARGETS ---
# Target 1: Magnitude (Actual log return of the denoised price)
df['target_mag'] = np.log(df['clean_close'].shift(-H) / df['clean_close'])

# Target 2: Direction (Binary classification based on magnitude)
df['target_dir'] = np.where(df['target_mag'] > 0, 1, 0)

# Target 3: Volatility (Standard deviation of the next H bars)
# We shift by -H so the model predicts the *future* volatility
df['target_vol'] = df['close'].rolling(window=H).std().shift(-H)

df = df.dropna(subset=['target_mag', 'target_dir', 'target_vol'])

# Feature Routing
macro_cols = [c for c in df.columns if c.startswith('macro_emb_')]
exclude_cols = ['open', 'high', 'low', 'close', 'clean_close', 'volume', 'target_dir', 'target_mag', 'target_vol', 'active_session_name', 'markov_regime', 'fwd_return_1h', 'target_dir_1h']
tech_features = [c for c in df.columns if c not in exclude_cols + macro_cols]

print(f"✅ Master Tensor Ready: {len(df)} bars")
print(f"🎯 Targets Generated: Direction, Magnitude, Volatility (H={H}, S={S})")

# [Optional] Future LLM Hook: When you have LLM scalars, you will replace 'macro_cols' with ['hawkish_score', 'risk_on_score', 'vol_expectation'] here.

🌊 Applying Wavelet Denoising to Price Action...
✅ Master Tensor Ready: 99981 bars
🎯 Targets Generated: Direction, Magnitude, Volatility (H=4, S=32)


/Users/mishazuy/Desktop/trading_proj/spider_global/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
# [Markdown]
# ### 2. Train the Multi-Task Engine
# We optimize the model using a combined loss function.

# 1. Split & Scale
split_idx = int(len(df) * 0.8)
train_df, val_df = df.iloc[:split_idx], df.iloc[split_idx:]

pca = PCA(n_components=8, random_state=42)
macro_train = pca.fit_transform(train_df[macro_cols])
macro_val = pca.transform(val_df[macro_cols])

X_train_raw = np.hstack([train_df[tech_features].values, macro_train])
X_val_raw = np.hstack([val_df[tech_features].values, macro_val])

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

# Extract Targets
y_dir_tr, y_mag_tr, y_vol_tr = train_df['target_dir'].values, train_df['target_mag'].values, train_df['target_vol'].values
y_dir_v, y_mag_v, y_vol_v = val_df['target_dir'].values, val_df['target_mag'].values, val_df['target_vol'].values

pos_weight = torch.tensor([(y_dir_tr == 0).sum() / (y_dir_tr == 1).sum()], dtype=torch.float32).to(device)

# Loaders
train_loader = DataLoader(MultiTaskDataset(X_train_scaled, y_dir_tr, y_mag_tr, y_vol_tr, S), batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(MultiTaskDataset(X_val_scaled, y_dir_v, y_mag_v, y_vol_v, S), batch_size=256, shuffle=False)

# 2. Model Setup
model = MultiTaskQuantformer(num_features=X_train_scaled.shape[1]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Loss Functions
bce_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
mse_loss = nn.MSELoss()

print("🚀 Training Multi-Task PnL Engine...")
best_val_loss = float('inf')
patience, no_improve = 5, 0

for epoch in range(50):
    model.train()
    for X_b, y_d, y_m, y_v in train_loader:
        X_b, y_d, y_m, y_v = X_b.to(device), y_d.to(device).unsqueeze(1), y_m.to(device).unsqueeze(1), y_v.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        
        # Forward pass yields 3 outputs
        pred_d, pred_m, pred_v = model(X_b)
        
        # The Custom Financial Loss Blend
        # 1. Directional Accuracy (Standard)
        loss_dir = bce_loss(pred_d, y_d)
        # 2. Magnitude Accuracy (Scale up by 100 to match BCE gradients)
        loss_mag = mse_loss(pred_m, y_m) * 100 
        # 3. Volatility Accuracy (Scale up to match gradients)
        loss_vol = mse_loss(pred_v, y_v) * 100 
        
        # Total Loss (You can weigh these differently later)
        total_loss = loss_dir + loss_mag + loss_vol
        
        total_loss.backward()
        optimizer.step()
        
    # Validation Loop
    model.eval()
    val_losses, y_true_d, y_pred_d = [], [], []
    with torch.no_grad():
        for X_b, y_d, y_m, y_v in val_loader:
            X_b, y_d, y_m, y_v = X_b.to(device), y_d.to(device).unsqueeze(1), y_m.to(device).unsqueeze(1), y_v.to(device).unsqueeze(1)
            pred_d, pred_m, pred_v = model(X_b)
            
            v_loss = bce_loss(pred_d, y_d) + (mse_loss(pred_m, y_m) * 100) + (mse_loss(pred_v, y_v) * 100)
            val_losses.append(v_loss.item())
            
            y_true_d.extend(y_d.cpu().numpy())
            y_pred_d.extend(torch.sigmoid(pred_d).cpu().numpy())
            
    avg_loss = np.mean(val_losses)
    auc = roc_auc_score(y_true_d, y_pred_d)
    
    print(f"Epoch {epoch+1} | Combined Val Loss: {avg_loss:.4f} | Directional AUC: {auc:.4f}")
    
    if avg_loss < best_val_loss:
        best_val_loss = avg_loss
        no_improve = 0
        torch.save(model.state_dict(), '../data/processed/core_quantformer.pth')
        joblib.dump(scaler, '../data/processed/core_scaler.pkl')
        joblib.dump(pca, '../data/processed/core_pca.pkl')
        joblib.dump(tech_features, '../data/processed/core_features.pkl')
    else:
        no_improve += 1
        if no_improve >= patience:
            print("🛑 Early stopping triggered.")
            break

print("✅ Multi-Task Artifacts Exported Successfully.")

🚀 Training Multi-Task PnL Engine...
Epoch 1 | Combined Val Loss: 1.4479 | Directional AUC: 0.6101
Epoch 2 | Combined Val Loss: 1.0516 | Directional AUC: 0.6164
Epoch 3 | Combined Val Loss: 0.9709 | Directional AUC: 0.6295
Epoch 4 | Combined Val Loss: 0.9494 | Directional AUC: 0.6364
Epoch 5 | Combined Val Loss: 0.9416 | Directional AUC: 0.6408
Epoch 6 | Combined Val Loss: 0.9374 | Directional AUC: 0.6444
Epoch 7 | Combined Val Loss: 0.9337 | Directional AUC: 0.6471
Epoch 8 | Combined Val Loss: 0.9281 | Directional AUC: 0.6512
Epoch 9 | Combined Val Loss: 0.9139 | Directional AUC: 0.6545
Epoch 10 | Combined Val Loss: 0.8812 | Directional AUC: 0.6558
Epoch 11 | Combined Val Loss: 0.8619 | Directional AUC: 0.6644
Epoch 12 | Combined Val Loss: 0.8528 | Directional AUC: 0.6737
Epoch 13 | Combined Val Loss: 0.8462 | Directional AUC: 0.6832
Epoch 14 | Combined Val Loss: 0.8397 | Directional AUC: 0.6923
Epoch 15 | Combined Val Loss: 0.8338 | Directional AUC: 0.7008
Epoch 16 | Combined Val Loss